# Numerical companion to *Kolyvagin's system of Gauss sums* by Rubin

This notebook can be considered as a numerical companion to K. Rubin, *Kolyvagin's system of Gauss sums*, in: Arithmetic Algebraic Geometry (Texel, 1989), Progr. Math. 89, Birkhäuser (1991), 309–324.

**Requirements.** A PARI/GP Jupyter kernel (e.g. the `pari_jupyter` package), and the two script files in the notebook directory:

* `scripts_Gausssums_v5.gp` ;
* `verify_stickelberger_rubin_2_.gp`.

In [ ]:
default(parisize, 2*10^9);
read("scripts_Gausssums_v5.gp");          \\ sets p = 37, delta = 2, BPREC = 4, loads all machinery
read("verify_stickelberger_rubin_2_.gp");
print("p = ", p, ",  delta = ", delta, " (primitive root mod p),  BPREC = ", BPREC);

p = 37,  delta = 2 (primitive root mod p),  BPREC = 4


## 1. The Herbrand–Ribet theorem

Let $p$ be an odd prime, $F=\mathbb{Q}(\mu_p)$, $\Delta=\operatorname{Gal}(F/\mathbb{Q})\cong(\mathbb{Z}/p\mathbb{Z})^\times$, and let $A$ be the $p$-part of the ideal class group of $F$.

Let $\omega:\Delta\to\mathbb{Z}_p^\times$ be the Teichmüller character. Since $\#\Delta=p-1$ is prime to $p$, $A$ splits into eigenspaces

$$A=\bigoplus_{i \bmod (p-1)} A^{\omega^i},\qquad A^{\chi}=e_\chi A,\qquad e_\chi=\frac{1}{p-1}\sum_{\sigma\in\Delta}\chi^{-1}(\sigma)\,\sigma .$$

**Theorem (Herbrand 1932; Ribet 1976).** For odd $i$ with $3\le i\le p-2$:

$$A^{\omega^i}\neq 0 \iff p \mid B_{p-i} \left( \iff p\mid B_{1,\omega^{-i}}\right),$$

where $B_{1,\psi}=\frac1p\sum_{a=1}^{p-1}a\,\psi(a)$, and the last equivalence is the congruence $B_{1,\omega^{j}}\equiv B_{j+1}/(j+1)\pmod p$ for $j\not\equiv 0,-1 \pmod{p-1}$.

Herbrand's direction ($\Rightarrow$) is a corollary of Stickelberger's theorem (§4 below). We have already discussed an approach to Ribet’s converse to Herbrand’s theorem using cyclotomic units. Kolyvagin's Euler system of Gauss sums, in Rubin's exposition, reproves the converse and in fact gives the exact order $\#A^{\omega^i}=p^{\operatorname{ord}_p B_{1,\omega^{-i}}}$.

**Example $p=37$.** The only even $j\le p-3$ with $37\mid B_j$ is $j=32$, so $(37,32)$ is the unique irregular pair, $i=p-j=5$ is the unique irregular odd index, and in fact $A=A^{\omega^5}\cong\mathbb{Z}/37\mathbb{Z}$ (the class number of $\mathbb{Q}(\zeta_{37})$ is exactly $37$).

In [ ]:
{
    print("even j with 37 | numerator(B_j),  2 <= j <= p-3:");
    for(j = 2, p - 3,
        if(j % 2 == 0 && Mod(numerator(bernfrac(j)), p) == 0,
            print("   irregular pair (37, ", j, ")")));
    print("B_32 = ", bernfrac(32));
    print("numerator(B_32)/37 = ", numerator(bernfrac(32))/37);
    print("B_{1,omega^{-5}} = B_{1,omega^{31}} mod 37 = ", lift(B1_OmegaPow(31, 37)));
    print("check  B_{1,omega^{31}} = B_32/32  (mod 37):  ",
          if(B1_OmegaPow(31, 37) == Mod(bernfrac(32)/32, 37), "OK", "FAIL"));
}

even j with 37 | numerator(B_j),  2 <= j <= p-3:
   irregular pair (37, 32)
B_32 = -7709321041217/510
numerator(B_32)/37 = -208360028141
B_{1,omega^{-5}} = B_{1,omega^{31}} mod 37 = 0
check  B_{1,omega^{31}} = B_32/32  (mod 37):  OK


## 2. Gauss sums

Fix an auxiliary prime $r\equiv 1 \pmod{pn}$; below $n=1$ and $r=149=1+4\cdot37$. Let $g_0$ = `znprimroot(r)` and let $\varepsilon=\varepsilon_{n,t}$ be the character of $(\mathbb{Z}/r\mathbb{Z})^\times$ of order $pn$ normalized by $\varepsilon(g_0)=\zeta_{pn}^{-1}$; equivalently $\varepsilon(a)\equiv a^{-(r-1)/pn}\pmod{t}$, where $t\mid r$ is the prime of $\mathbb{Q}(\zeta_{pn})$ singled out in §3. Rubin's Gauss sum is

$$g \;=\; g(n,t,\zeta_r)\;=\;\sum_{a=1}^{r-1}\varepsilon(a)\,\zeta_r^{\,a}\ \in\ \mathbb{Q}(\zeta_{pnr}).$$

Elementary properties: $g\,\bar g = r$, and $\sigma_c(g)=\varepsilon(c)^{-1}g$ for every $c\equiv1\pmod{pn}$. Since $\operatorname{Gal}\!\big(\mathbb{Q}(\zeta_{pnr})/\mathbb{Q}(\zeta_{pn})\big)=\{\sigma_c: c\equiv 1 \bmod pn\}$ and $\varepsilon$ takes values in $\mu_{pn}$, the power $g^{pn}$ is invariant under this group, hence

$$g^{pn}\in\mathbb{Q}(\zeta_{pn}).$$

The function `Cyclo_Subfield(g, N, m)` proves such membership constructively. It raises an error whenever the element is not in the subfield — so a successful return is the verification.

In [ ]:
{
    n = 1;  r = 149;              \\ auxiliary prime:  r = 1 (mod p*n)
    m = p*n;  N = p*n*r;          \\ g lives in Q(zeta_N),  N = 5513,  phi(N) = 5328
    
    g = Gauss_Sum_Fast(n, r);
    print("g = g(1, t, zeta_149) computed in Q(zeta_", N, ")");

    \\ |g|^2 = r :  complex conjugation is tau_{-1} = tau_{N-1}
    print("g * conj(g) = r :                ",
          if(Tau_Cyclo(g, N, N-1) * g == r, "OK", "FAIL"));

    \\ sigma_c(g) = eps(c)^{-1} g  for c = 1 (mod pn);  test c = 38
    my(zpn    = Mod(x, polcyclo(N))^r);                    \\ zeta_37 inside Q(zeta_N)
    my(epsinv = zpn^znlog(Mod(38, r), znprimroot(r)));     \\ eps(38)^{-1}
    print("sigma_38(g) = eps(38)^{-1} g :   ",
          if(Tau_Cyclo(g, N, 38) == epsinv * g, "OK", "FAIL"));
}

g = g(1, t, zeta_149) computed in Q(zeta_5513)
g * conj(g) = r :                OK
sigma_38(g) = eps(38)^{-1} g :   OK


In [ ]:
{
    \\ the p-th power of the Gauss sum lies in the cyclotomic subfield Q(zeta_37):
    gettime();
    gp37 = g^m;                          \\ g^37, still written in Q(zeta_5513)
    print("g^", m, " computed ");
    A = Cyclo_Subfield(gp37, N, m);      \\ errors if g^37 were NOT in Q(zeta_37)
    print("descent  g^", m, " -> Q(zeta_", m, ")  done ");
    print("");
    print("g^", m, " = ", A);
}

g^37 computed 
descent  g^37 -> Q(zeta_37)  done 

g^37 = Mod(-1956324895244093787509545180453163110142*x^35 - 3277884097367146478637955719975970074263*x^34 - 5624584898180734096824515727058393986739*x^33 - 3206571865979983392841897358642050611631*x^32 - 1066932411454564884255525501212424614077*x^31 + 555547237078726479223831893780745744599*x^30 - 1924324353391810170035703282489681976854*x^29 - 1778678944893889945703760009756859308801*x^28 - 1243937657412489969926383820257865148848*x^27 - 2287078035202070139202578159859708274773*x^26 + 2007962313171970282398562424136750888869*x^25 - 765633108277120029515176751993902936270*x^24 - 1267146588463334305653542177211062836817*x^23 - 2729646055055037998299503848090328199465*x^22 - 1849904931876691747328775782377657759240*x^21 - 3971961343619934264877447368325780048349*x^20 - 126212966997487579453920256100932915346*x^19 - 862813904524121303312736441503901134624*x^18 - 3048481563886201556455955130631834986523*x^17 - 50317238853407176434452739390

In [ ]:
{
    \\ negative control: g itself does NOT lie in Q(zeta_37),
    \\ and Cyclo_Subfield correctly refuses to descend it:
    iferr(Cyclo_Subfield(g, N, m),
          E,
          print("as expected, Cyclo_Subfield raises an error:  g is not in Q(zeta_37)"));
}

as expected, Cyclo_Subfield raises an error:  g is not in Q(zeta_37)


## 3. Stickelberger's theorem

For the modulus $pn$ put $G=\operatorname{Gal}(\mathbb{Q}(\zeta_{pn})/\mathbb{Q})$, $\tau_a:\zeta\mapsto\zeta^a$, and define

$$\theta \;=\!\!\sum_{\substack{a\bmod pn\\ (a,pn)=1}}\!\Big\{\frac{a}{pn}\Big\}\,\tau_a^{-1}\ \in\ \mathbb{Q}[G],
\qquad
s(n)\;=\;pn\,\theta\;=\!\!\sum_{\substack{a\bmod pn\\(a,pn)=1}}\!a\,\tau_a^{-1}\ \in\ \mathbb{Z}[G].$$

**Theorem (Stickelberger).** If $\beta\in\mathbb{Z}[G]$ and $\beta\theta\in\mathbb{Z}[G]$, then $\beta\theta$ annihilates the ideal class group of $\mathbb{Q}(\zeta_{pn})$.

**Example.** $s(n)$ annihilates the ideal class group of $\mathbb{Q}(\zeta_{pn})$.

The engine of the proof is the classical ideal factorization of Gauss sums with $t$ the prime of $\mathbb{Q}(\zeta_{pn})$ above $r$ determined by $\varepsilon$,

$$\big(g^{pn}\big) \;=\; t^{\,s(n)} \;=\; \prod_{(c,pn)=1}\tau_c(t)^{\ \operatorname{lift}(c^{-1}\bmod pn)} .$$

The verification below identifies each conjugate $\tau_c(t)$ through its residue field: with $u = g_0^{(r-1)/pn}\bmod r$ one has $\zeta_{pn}\equiv u\pmod t$, hence $\zeta_{pn}\equiv u^{\,c^{-1}}\pmod{\tau_c(t)}$, and then compares valuations.

In [ ]:
{
    S1 = Stickelberger(1);      \\ s(1) = sum_{a mod 37} a * tau_{a^{-1}}   (t_MAP)
    print("s(1):  coefficient at tau_c  (equals lift(c^{-1} mod 37)),  c = 1..36:");
    print(vector(m - 1, c, GR_Coeff(S1, c)));
}

s(1):  coefficient at tau_c  (equals lift(c^{-1} mod 37)),  c = 1..36:
[1, 19, 25, 28, 15, 31, 16, 14, 33, 26, 27, 34, 20, 8, 5, 7, 24, 35, 2, 13, 30, 32, 29, 17, 3, 10, 11, 4, 23, 21, 6, 22, 9, 12, 18, 36]


In [ ]:
{
    K   = nfinit(polcyclo(m));                      \\ Q(zeta_37) as a number field
    dec = idealprimedec(K, r);                      \\ 149 = 1 (mod 37) splits completely
    u   = lift(Mod(znprimroot(r), r)^((r-1)/m));    \\ the prime t:  zeta_37 = u  (mod t)
    print("149 splits into ", #dec, " primes;  t is fixed by  zeta_37 = ", u, "  (mod t)");

    my(fa = idealfactor(K, A), ok0 = 1);
    for(i = 1, matsize(fa)[1], if(fa[i,1].p != r, ok0 = 0));
    print("(g^37) supported only above r:                        ", if(ok0, "OK", "FAIL"));

    my(ok = 1);
    for(c = 1, m - 1,
        my(ci = lift(Mod(c, m)^(-1)));
        my(w  = lift(Mod(u, r)^ci));                \\ zeta_37 = u^{c^{-1}}  (mod tau_c(t))
        my(P  = Prime_With_Residue(K, dec, r, w));
        if(nfeltval(K, A, P) != ci,
            ok = 0; print("   mismatch at c = ", c)));
    print("v_{tau_c(t)}(g^37) = lift(c^{-1} mod 37) for all c:   ", if(ok, "OK", "FAIL"));
    print("i.e.  (g^37) = t^{s(1)}   -- classical Stickelberger factorization");
}

149 splits into 36 primes;  t is fixed by  zeta_37 = 16  (mod t)
(g^37) supported only above r:                        OK
v_{tau_c(t)}(g^37) = lift(c^{-1} mod 37) for all c:   OK
i.e.  (g^37) = t^{s(1)}   -- classical Stickelberger factorization


## 4. Herbrand's theorem as a corollary of Stickelberger's theorem

Take $\beta = c-\tau_c$, so $\beta\theta\in\mathbb{Z}[G]$ and annihilates the class group. Now project to the $\chi$-eigenspace for an odd $\chi=\omega^k\ne\omega$. A direct computation gives $e_\chi\theta = B_{1,\chi^{-1}}\,e_\chi$, so $(c-\chi(c))\,B_{1,\chi^{-1}}$ annihilates $A^{\chi}$; choosing $c=\delta$ a primitive root makes $c-\chi(c)\equiv\delta-\delta^{k}\not\equiv0\pmod p$ (as $k\neq1$), a unit. Hence

$$B_{1,\chi^{-1}}\ \text{annihilates}\ A^{\chi}
\qquad\Longrightarrow\qquad
A^{\omega^i}\neq0 \Rightarrow p\mid B_{1,\omega^{-i}} \equiv \frac{B_{p-i}}{p-i}\ (\mathrm{mod}\ p),$$

which is exactly Herbrand's theorem.

In the scripts, the integralized element is $\theta(1)=(\sigma_\delta-\delta)\theta$ and its $\chi$-projection is the scalar

$$e_\chi\,\theta(1) \;=\; \big(\chi(\delta)-\delta\big)\,B_{1,\chi^{-1}}\;e_\chi \;=:\; \delta(1)\,e_\chi ,
\qquad \chi(\delta)-\delta\in\mathbb{Z}_p^\times \ \ (\chi\neq\omega),$$

which is Rubin's derived Stickelberger element $\delta(1)$.

The table below lists $B_{1,\omega^{-k}}$ and $\delta(1)$ mod $37$ for every odd $k$: both vanish only at $k=5$. So at the irregular index Stickelberger only says "$0$ annihilates $A^{\omega^5}$" — an empty statement. The Stickelberger ideal alone cannot bound $A^{\omega^5}$; that is exactly the problem Kolyvagin's derivative classes solve (§§6–7).

In [ ]:
{
    print("chi = omega^k, k odd:   B_{1,chi^{-1}} mod 37   delta(1) = (chi(delta)-b)*B_{1,chi^{-1}} mod 37");
    for(k = 3, p - 2,
        if(k % 2,
            printf("   k = %2d:                %2d                          %2d\n",
                   k, lift(B1_OmegaPow(-k, 37)), lift(Delta_Fast(1, k, 37)))));
}

chi = omega^k, k odd:   B_{1,chi^{-1}} mod 37   delta(1) = (chi(delta)-b)*B_{1,chi^{-1}} mod 37
   k =  3:                24                          33
   k =  5:                 0                           0
   k =  7:                 5                           1
   k =  9:                 5                          34
   k = 11:                 9                          25
   k = 13:                30                          20
   k = 15:                36                          16
   k = 17:                10                          12
   k = 19:                12                          26
   k = 21:                 6                          14
   k = 23:                34                          28
   k = 25:                23                           7
   k = 27:                30                           9
   k = 29:                 2                           7
   k = 31:                21                          13
   k = 33:                 4                     

## 5. Rubin's version of Stickelberger's theorem

Rubin removes the exponent $pn$ from the classical factorization. Choose $b_n\in\mathbb{Z}$ with $\zeta^{\sigma_\delta}=\zeta^{\,b_n}$ for all $\zeta\in\mu_{Mpn}$ and $b_n\equiv1\pmod n$, and set

$$\alpha(n,t)\;=\;g(n,t,\zeta_r)^{\,\sigma_\delta-b_n},
\qquad
\theta(n)\;=\;\tfrac{1}{pn}\,(\sigma_\delta-b_n)\,s(n)\ \in\ \mathbb{Z}[G].$$

**Proposition 1.1 (Rubin).** *(i)* $\alpha(n,t)\in \mathbb{Q}(\zeta_{pn})^\times$; *(ii)* as fractional ideals of $\mathbb{Q}(\zeta_{pn})$,

$$\big(\alpha(n,t)\big)\;=\;\theta(n)\,t\;=\;\prod_{(c,pn)=1}\tau_c(t)^{\,\theta_c},
\qquad
\theta_c=-\Big\lfloor \tfrac{b_n\,\operatorname{lift}(c^{-1}\bmod pn)}{pn}\Big\rfloor .$$

$\theta(n)$ is integral and $\theta(n)\,t$ is exhibited as principal with an explicit Gauss-sum generator. 

Below we verify (i) and (ii) at $p=37$, $n=1$, $r=149$ by comparing $v_{\tau_c(t)}(\alpha^{37})$ with $37\,\theta_c$ at all 36 conjugates, plus the support and norm consistency checks.

In [ ]:
{
    d    = delta_pn(n);        \\ sigma_delta as an element of (Z/37Z)^*:  d = 2
    beta = b(n);               \\ b_1 = delta mod 37^BPREC:  beta = 2
    th   = Theta_Element(n);   \\ theta(1) = (sigma_delta - b_1) * theta   (integral!)
    print("theta(1):   (c = 1..36:)");
    print(vector(m - 1, c, GR_Coeff(th, c)));
    print("");

    Am = Tau_Cyclo(A, m, d) * A^(-beta);        \\ = alpha(1,t)^37  in Q(zeta_37)

    my(fa = idealfactor(K, Am), ok0 = 1);
    for(i = 1, matsize(fa)[1], if(fa[i,1].p != r, ok0 = 0));
    print("(alpha^37) supported only above r:                 ", if(ok0, "OK", "FAIL"));

    my(ok = 1, S = 0);
    for(c = 1, m - 1,
        my(ci = lift(Mod(c, m)^(-1)));
        my(w  = lift(Mod(u, r)^ci));
        my(P  = Prime_With_Residue(K, dec, r, w));
        my(tc = GR_Coeff(th, c));
        S += tc;
        if(nfeltval(K, Am, P) != m*tc,
            ok = 0; print("   mismatch at c = ", c)));
    print("v_{tau_c(t)}(alpha^37) = 37*theta_c for all c:     ", if(ok, "OK", "FAIL"));
    print("i.e.  (alpha(1,t)) = theta(1) t   -- Rubin, Proposition 1.1(ii)");
    print("Norm(alpha^37) = r^(37 * sum_c theta_c):           ",
          if(idealnorm(K, Am) == abs(r^(m*S)), "OK", "FAIL"));

    \\ Optional (one expensive inversion in degree phi(5513) = 5328):
    \\ verify that alpha itself, not only alpha^37, lies in Q(zeta_37):
    
    ebig  = lift(chinese(Mod(d, m), Mod(1, r)));
    alpha = Tau_Cyclo(g, N, ebig) * g^(-beta);
    al    = Cyclo_Subfield(alpha, N, m);       \\ errors if alpha were not there
    print("alpha in Q(zeta_37), consistent with alpha^37:  ", al^m == Am);
}

theta(1):  coefficient at tau_c is  -floor(2*c^{-1}/37),  c = 1..36:
[0, -1, -1, -1, 0, -1, 0, 0, -1, -1, -1, -1, -1, 0, 0, 0, -1, -1, 0, 0, -1, -1, -1, 0, 0, 0, 0, 0, -1, -1, 0, -1, 0, 0, 0, -1]

(alpha^37) supported only above r:                 OK
v_{tau_c(t)}(alpha^37) = 37*theta_c for all c:     OK
i.e.  (alpha(1,t)) = theta(1) t   -- Rubin, Proposition 1.1(ii)
Norm(alpha^37) = r^(37 * sum_c theta_c):           OK
alpha in Q(zeta_37), consistent with alpha^37:  1


## 6. The modified Stickelberger theorem (Rubin, Thm. 4.1 and Cor. 4.2)

Let $M=p^s$ and let $n$ be squarefree with every prime factor $\ell\equiv1\pmod M$. For $\ell\mid n$ fix a generator $\sigma_\ell$ of $G_\ell=\operatorname{Gal}\big(\mathbb{Q}(\zeta_{pn})/\mathbb{Q}(\zeta_{pn/\ell})\big)$ and set

$$D_\ell=\sum_{i=1}^{\ell-2} i\,\sigma_\ell^{\,i},
\qquad D_n=\prod_{\ell\mid n}D_\ell,
\qquad (\sigma_\ell-1)\,D_\ell=(\ell-1)-N_\ell,\quad N_n=\sum_{\tau\in G_n}\tau .$$

**Derived Stickelberger (Rubin, Cor. 2.6).** The Kolyvagin derivative collapses the $\chi$-projected Stickelberger element onto the norm element:

$$D_n\,\theta(n,\chi)\;\equiv\;\delta(n)\; e_\chi N_n \pmod{M},
\qquad \theta(n,\chi)=e_\chi\,\theta(n),$$

which **defines** the scalar $\delta(n)\in\mathbb{Z}/M\mathbb{Z}$. Following Rubin's §4, let $d(n)$ be the largest divisor of $M$ dividing $\delta(n)$; by §4 above, $d(1)(\mathbb{Z}/M\mathbb{Z})=B_{1,\chi^{-1}}(\mathbb{Z}/M\mathbb{Z})$.

**Theorem 4.1 (Rubin).** Assume $d(1)\,\#(A^\chi)\,p\mid M$, let $n\in S$, let $B\subseteq A$ be the subgroup generated by the classes of primes dividing $n$, and let $c\in A^\chi$ with $c\notin B^\chi$ and $d(n)\mid d(1)$. Then there is a prime $\lambda$ of $F$ above some $\ell\in S$ such that 

(i) the class of $\lambda$ projects to $c$ in $A^\chi$,

(ii) $d(n\ell)\mid d(n)$, and $d(n)/d(n\ell)$ annihilates $c$ in $A^\chi/B^\chi$.

**Corollary 4.2 (Rubin).** Under the same hypothesis on $M$: $\#\big(A^\chi/B^\chi\big)\ \big|\ d(n)$ for every $n\in S$. 

In particular ($n=1$): 

* $\#A^\chi \mid d(1)$, and $d(1)\sim B_{1,\chi^{-1}}$. 

* Combined with the analytic class number formula this yields $\#A^\chi=p^{\operatorname{ord}_p B_{1,\chi^{-1}}}$ for all odd $\chi\ne\omega$ — in particular Ribet's converse to Herbrand.

Below, $\delta(149)$ is computed step by step for $\chi=\omega^5$, $M=37$, exactly as the definition reads. The steps are: 

**0.** form coefficients of $\theta(149)$; 

**1.** project by $e_\chi$; 

**2.** apply $D_{149}$; 

**3.** divide by $e_\chi N_{149}$.


## 7. Searching for nontrivial derived classes: auxiliary primes with $\delta(\ell)\not\equiv0$

At $p=37$, $\chi=\omega^5$ we found $\delta(1)\equiv0\pmod{37}$, i.e. $d(1)=M$: at level 1 the Stickelberger relation degenerates and gives no bound on $A^{\omega^5}$. The Kolyvagin system recovers the lost information one level up: take auxiliary primes $\ell\equiv1\pmod M$ and compute $\delta(\ell)$.

If $\delta(\ell)$ is a unit mod $p$, then $d(\ell)=1$. Since $\operatorname{ord}_{37}B_{1,\omega^{31}}=1$, the machinery gives $\#A^{\omega^5}\mid 37$, and with Herbrand $A^{\omega^5}=\mathbb{Z}/37\mathbb{Z}$. In Rubin's notation, $d(n_1)=1$ already for the first auxiliary prime — e.g. $n_1=149$.

**Correctness.** With $M=37$ the running hypothesis $d(1)\,\#(A^\chi)\,p\mid M$ of Thm. 4.1/Cor. 4.2 is not satisfied — a rigorous run of the machine needs $M\ge 37^3$ and primes $\ell\equiv1\pmod{np^2M}$. The mod-37 computation below exhibits precisely the phenomenon the proof exploits: the derivative recovers at level $\ell$ the unit that vanished at level 1. (An optional line runs one example mod $37^2$, where auxiliary primes must satisfy $\ell\equiv1\pmod{37^2}$.)

The choice of generators $\sigma_\ell=$ `znprimroot(l)` changes each $\delta(\ell)$ by a unit; only the vanishing/nonvanishing — that is, $d(\ell)$ — is canonical.

In [1]:
Search_Nontrivial(k, M, lmax) =
{
    my(found = List());
    print("chi = omega^", k, ",  M = ", M,
          ":  scanning primes l = 1 (mod ", M, "),  l <= ", lmax);
    forprime(l = 3, lmax,
        if(l != p && (l - 1) % M == 0,
            my(dl = lift(Delta_Fast(l, k, M)));
            print("   l = ", l, ":   delta(l) = ", dl,
                  if(dl % p, "    <-- unit mod p:  nontrivial derived class", ""));
            if(dl % p, listput(found, [l, dl]))));
    return(Vec(found));
}

In [1]:
{
    good = Search_Nontrivial(5, 37, 1500);
    print("");
    print("auxiliary primes with delta(l) a unit mod 37:  ", good);
    print("");
    print("B_{1,omega^{31}} mod 37^2 = ", lift(B1_OmegaPow(31, 37^2)),
          "   (a nonzero multiple of 37  =>  ord_37 = 1  =>  #A^{omega^5} = 37)");

    \\ Optional, mod 37^2 (auxiliary primes must then be = 1 mod 37^2 = 1369; slower):
    print("delta(5477) mod 37^2 = ", lift(Delta_Fast(5477, 5, 37^2)));
}

chi = omega^5,  M = 37:  scanning primes l = 1 (mod 37),  l <= 1500
   l = 149:   delta(l) = 14    <-- unit mod p:  nontrivial derived class
   l = 223:   delta(l) = 22    <-- unit mod p:  nontrivial derived class
   l = 593:   delta(l) = 6    <-- unit mod p:  nontrivial derived class
   l = 1259:   delta(l) = 13    <-- unit mod p:  nontrivial derived class
   l = 1481:   delta(l) = 5    <-- unit mod p:  nontrivial derived class

auxiliary primes with delta(l) a unit mod 37:  [[149, 14], [223, 22], [593, 6], [1259, 13], [1481, 5]]

B_{1,omega^{31}} mod 37^2 = 851   (a nonzero multiple of 37  =>  ord_37 = 1  =>  #A^{omega^5} = 37)
delta(5477) mod 37^2 = 720


Even if $\delta(\ell)$ is not a unit, the corresponding class may still be non-trivial.